In [46]:
import pandas as pd
import re
import random

In [47]:
def stringToLabel(head_labels):
    ct = {}
    for i in range(len(head_labels)):
        ct['column' + str(i+1)] = head_labels[i]
    return ct

In [48]:
head_labels = 'Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,UniqueCarrier,AirlineID,Carrier,TailNum,FlightNum,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime'.split(',')
columnNameToLabel = stringToLabel(head_labels)

In [49]:
def lineToDataStruct(lineElements, columnNameToLabel):

    index_dependant = 0
    for element in lineElements:
        if element.startswith('"dependant":'):
            break
        index_dependant += 1

    rhs = lineElements[index_dependant + 1].split(':')[1]
    for removeChar in ['"','[',']','{','}']:
        rhs = rhs.replace(removeChar, '')
    determinant_index = 0
    rhs = columnNameToLabel[rhs]

    for element in lineElements:
        if element.startswith('"determinant'):
            #print('found at ' + str(determinant_index))
            break
        determinant_index += 1

    X = [] # RHS
    for s in lineElements[determinant_index + 1: index_dependant]:
        match = re.search(r'"columnIdentifier":"(column\d+)"', s)
        if match:
            result = match.group(1)
            X.append(columnNameToLabel[result])

    return X,rhs

In [50]:
data = []
with open('2025-11-14_17-00-10_fds.csv', 'r') as file:
    for line in file:
        lhs, rhs = lineToDataStruct(line.strip().split(","), columnNameToLabel)
        prettyPrint = lambda lst: ', '.join(lst)
        #print(prettyPrint(lhs), '->',rhs)
        data .append( (lhs, rhs) )

In [51]:
ids = random.sample(range(len(data)), 100)

In [52]:
sample = [data[i] for i in ids]

In [53]:
sample[2]

(['OriginCityName',
  'DestCityMarketID',
  'DepartureDelayGroups',
  'DepTimeBlk',
  'DayofMonth',
  'CarrierDelay'],
 'ArrDel15')

In [54]:
def find_fds_for_rhs(target_rhs, data,sorted=False, lhs_validator = lambda x: True):
    results = []
    for tuple in data:
        lhs, rhs = tuple
        if rhs == target_rhs and lhs_validator(lhs):
            results.append(tuple)
    if sorted:
        results.sort(key=lambda x: len(x[0]))
    return results

In [55]:
genuine_fds = [find_fds_for_rhs('ArrDelay', data, sorted = True)[0]]

In [56]:
firsts = [find_fds_for_rhs(label, data, sorted = True)[0] for label in head_labels]

In [57]:
firsts

[(['ArrTime'], 'Year'),
 (['ArrTime'], 'Quarter'),
 (['ArrTime'], 'Month'),
 (['FlightDate'], 'DayofMonth'),
 (['DayofMonth'], 'DayOfWeek'),
 (['DayofMonth'], 'FlightDate'),
 (['ArrTime'], 'UniqueCarrier'),
 (['ArrTime'], 'AirlineID'),
 (['ArrTime'], 'Carrier'),
 (['FlightNum', 'CRSArrTime', 'DayofMonth'], 'TailNum'),
 (['CRSDepTime', 'CRSArrTime', 'DayofMonth'], 'FlightNum'),
 (['OriginAirportSeqID'], 'OriginAirportID'),
 (['OriginAirportID'], 'OriginAirportSeqID'),
 (['OriginAirportID'], 'OriginCityMarketID'),
 (['OriginAirportID'], 'Origin'),
 (['OriginAirportID'], 'OriginCityName'),
 (['OriginAirportID'], 'OriginState'),
 (['OriginAirportID'], 'OriginStateFips'),
 (['OriginAirportID'], 'OriginStateName'),
 (['OriginAirportID'], 'OriginWac'),
 (['DestAirportSeqID'], 'DestAirportID'),
 (['DestAirportID'], 'DestAirportSeqID'),
 (['DestAirportID'], 'DestCityMarketID'),
 (['DestAirportID'], 'Dest'),
 (['DestAirportID'], 'DestCityName'),
 (['DestAirportID'], 'DestState'),
 (['DestAirport

In [58]:
def create_sample(data, n):
    ids = random.sample(range(len(data)), n)
    sample = [data[i] for i in ids]
    return sample

In [59]:
import csv
import os

def export_sample_to_csv(filename):
    if not os.path.exists(filename):
        with open(filename, 'w', newline='') as f:
            writer = csv.writer(f)
            write_back = [['LHS', 'RHS']]
            write_back.extend(create_sample(data, 1000))
            writer.writerows(write_back)


In [60]:
export_sample_to_csv( 'training_data.csv')

In [61]:
find_fds_for_rhs('NASDelay', data, sorted = True)

[(['TailNum', 'DepTime', 'ArrTime'], 'NASDelay'),
 (['DepTime', 'CRSArrTime', 'ArrTime'], 'NASDelay'),
 (['DepTime', 'ArrTime', 'ArrDelay'], 'NASDelay'),
 (['DepTime', 'ArrTime', 'ArrDelayMinutes'], 'NASDelay'),
 (['DepTime', 'ArrTime', 'CarrierDelay'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'ActualElapsedTime'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'AirTime'], 'NASDelay'),
 (['TailNum', 'DepDelay', 'ArrTime'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'CarrierDelay'], 'NASDelay'),
 (['TailNum', 'DayofMonth', 'ArrTime'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'FlightDate'], 'NASDelay'),
 (['CRSArrTime', 'ArrTime', 'ActualElapsedTime'], 'NASDelay'),
 (['DepDelay', 'CRSArrTime', 'ArrTime'], 'NASDelay'),
 (['DepDelayMinutes', 'CRSArrTime', 'ArrTime'], 'NASDelay'),
 (['CRSArrTime', 'DayofMonth', 'ArrTime'], 'NASDelay'),
 (['CRSArrTime', 'ArrTime', 'FlightDate'], 'NASDelay'),
 (['ArrTime', 'ArrDelay', 'ActualElapsedTime'], 'NASDelay'),
 (['ArrTime', 'ArrDelayMinutes', 'ActualElapsedTime'], 'NASD

In [62]:
find_fds_for_rhs('FlightNum', data, sorted = True, lhs_validator = lambda lhs: any(['origin' in label.lower() for label in lhs]) and not any(['city' in label.lower() for label in lhs]))

[(['TailNum', 'OriginAirportID', 'CRSDepTime', 'CRSArrTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportSeqID', 'CRSDepTime', 'CRSArrTime'], 'FlightNum'),
 (['TailNum', 'Origin', 'CRSDepTime', 'CRSArrTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportID', 'CRSArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportSeqID', 'CRSArrTime', 'CRSElapsedTime'],
  'FlightNum'),
 (['TailNum', 'Origin', 'CRSArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportID', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginAirportSeqID', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'Origin', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginState', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginStateFips', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginStateName', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginWac', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginAirportID', 'DestAi

In [72]:

def build_lhs_validator(required_substrings, forbidden_substrings):
    def lhs_validator(lhs):
        lhs_lower = [label.lower() for label in lhs]
        if ( len(required_substrings) > 0):
            for req in required_substrings:
                if not any([req in label for label in lhs_lower]):
                    return False
        else: 
            print('awdsad')
        for forb in forbidden_substrings:
            if any([forb in label for label in lhs_lower]):
                return False
        return True
    return lhs_validator

In [73]:
print(build_lhs_validator(['origin'], ['time'])(['city_name', 'origin_state']))

True


In [75]:
find_fds_for_rhs('', data, sorted = True, lhs_validator = build_lhs_validator([], []))

[]

In [ ]:
len(data)

1525766

In [30]:
new_file_path = '2025-11-21_14-13-53_fds.csv'

In [37]:
new_labels = 'Car_id,Date,Customer Name,Gender,Annual Income,Dealer_Name,Company,Model,Engine,Transmission,Color,Price ($),Dealer_No ,Body Style,Phone,Dealer_Region'.split(',')

In [38]:
new_ct = stringToLabel(new_labels)

In [39]:
new_ct

{'column1': 'Car_id',
 'column2': 'Date',
 'column3': 'Customer Name',
 'column4': 'Gender',
 'column5': 'Annual Income',
 'column6': 'Dealer_Name',
 'column7': 'Company',
 'column8': 'Model',
 'column9': 'Engine',
 'column10': 'Transmission',
 'column11': 'Color',
 'column12': 'Price ($)',
 'column13': 'Dealer_No ',
 'column14': 'Body Style',
 'column15': 'Phone',
 'column16': 'Dealer_Region'}

In [40]:
skibidi = []
with open(new_file_path, 'r') as file:
    for line in file:
        print(line)
        lhs, rhs = lineToDataStruct(line.strip().split(","), new_ct)
        prettyPrint = lambda lst: ', '.join(lst)
        print(prettyPrint(lhs), '->',rhs)
        skibidi.append( (lhs, rhs) )

{"type":"FunctionalDependency","determinant":{"columnIdentifiers":[{"tableIdentifier":"car_sales.csv","columnIdentifier":"column1"}]},"dependant":{"tableIdentifier":"car_sales.csv","columnIdentifier":"column15"}}

Car_id -> Phone
{"type":"FunctionalDependency","determinant":{"columnIdentifiers":[{"tableIdentifier":"car_sales.csv","columnIdentifier":"column1"}]},"dependant":{"tableIdentifier":"car_sales.csv","columnIdentifier":"column3"}}

Car_id -> Customer Name
{"type":"FunctionalDependency","determinant":{"columnIdentifiers":[{"tableIdentifier":"car_sales.csv","columnIdentifier":"column1"}]},"dependant":{"tableIdentifier":"car_sales.csv","columnIdentifier":"column5"}}

Car_id -> Annual Income
{"type":"FunctionalDependency","determinant":{"columnIdentifiers":[{"tableIdentifier":"car_sales.csv","columnIdentifier":"column1"}]},"dependant":{"tableIdentifier":"car_sales.csv","columnIdentifier":"column12"}}

Car_id -> Price ($)
{"type":"FunctionalDependency","determinant":{"columnIdentifie

In [77]:
find_fds_for_rhs('', skibidi, sorted = True, lhs_validator = lambda x : 'phone' in x )

[]

In [ ]:
def sortSkibidi(dat, exclude_lhs):
    results = []
    for t in dat:
        if not any([ any([ exclude in l.lower() for exclude in exclude_lhs]) for l in t[0]]):
            results.append(t)
    results.sort(key=lambda x: len(x[0]))
    return results

In [ ]:
for t in sortSkibidi(skibidi, ['phone', 'car_id']):
    if len(t[0] )< 5:
        print(t)

(['Dealer_Name'], 'Dealer_No ')
(['Engine'], 'Transmission')
(['Transmission'], 'Engine')
(['Price ($)', 'Model'], 'Company')
(['Date', 'Customer Name', 'Model'], 'Company')
(['Date', 'Customer Name', 'Model'], 'Body Style')
(['Customer Name', 'Dealer_Name', 'Model'], 'Company')
(['Dealer_No ', 'Customer Name', 'Model'], 'Company')
(['Dealer_Region', 'Customer Name', 'Model'], 'Company')
(['Body Style', 'Customer Name', 'Model'], 'Company')
(['Date', 'Dealer_Name', 'Model'], 'Company')
(['Dealer_Region', 'Date', 'Model'], 'Company')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Car_id')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Phone')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Model')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Company')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Dealer_Name')
(['Price ($)', 'Date', 'Customer Name', 'Annual Income'], 'Dealer_No ')
(['Price ($)', 'Date', 'Customer Name', 'Annu

In [113]:
def export_fds_to_csv(filename, subset):
    if not os.path.exists(filename):
        with open(filename, 'w', newline='') as f:
            writer = csv.writer(f)
            write_back = [['LHS', 'RHS']]
            write_back.extend(subset)
            writer.writerows(write_back)

In [115]:
subset = [t for t in sortSkibidi(skibidi, ['phone', 'car_id']) if len(t[0]) < 5]

In [116]:
filename = 'cars_relevant_fds'

export_fds_to_csv(filename, subset)

In [118]:
find_fds_for_rhs('Car_id', skibidi, sorted=True)

[(['Phone', 'Customer Name'], 'Car_id'),
 (['Price ($)', 'Phone'], 'Car_id'),
 (['Phone', 'Date'], 'Car_id'),
 (['Phone', 'Annual Income', 'Model'], 'Car_id'),
 (['Phone', 'Annual Income', 'Dealer_Name'], 'Car_id'),
 (['Dealer_No ', 'Phone', 'Annual Income'], 'Car_id'),
 (['Phone', 'Dealer_Region', 'Annual Income'], 'Car_id'),
 (['Phone', 'Dealer_Name', 'Model'], 'Car_id'),
 (['Dealer_No ', 'Phone', 'Model'], 'Car_id'),
 (['Phone', 'Dealer_Region', 'Model'], 'Car_id'),
 (['Phone', 'Gender', 'Model'], 'Car_id'),
 (['Phone', 'Dealer_Name', 'Company'], 'Car_id'),
 (['Dealer_No ', 'Phone', 'Company'], 'Car_id'),
 (['Phone', 'Dealer_Region', 'Company'], 'Car_id'),
 (['Body Style', 'Phone', 'Annual Income', 'Company'], 'Car_id'),
 (['Color', 'Phone', 'Annual Income', 'Company'], 'Car_id'),
 (['Color', 'Body Style', 'Phone', 'Annual Income'], 'Car_id'),
 (['Color', 'Phone', 'Annual Income', 'Engine'], 'Car_id'),
 (['Transmission', 'Color', 'Phone', 'Annual Income'], 'Car_id'),
 (['Body Style'